# Préparation des données — California Housing

Pipeline de préparation pour la modélisation :
1. Lecture depuis la base DuckDB (`data/raw/california_housing.db`)
2. Séparation features / cible
3. Train/test split (80/20, seed fixe pour reproductibilité)
4. Standardisation des features (`StandardScaler`, fit sur le train uniquement)
5. Persistance dans `data/processed/california_housing_processed.db` + sauvegarde du scaler

In [1]:
from pathlib import Path

import duckdb
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

TARGET = "MedHouseVal"
RAW_DB = Path("../data/raw/california_housing.db")
PROCESSED_DB = Path("../data/processed/california_housing_processed.db")
SCALER_PATH = Path("../models/scaler.joblib")
RANDOM_STATE = 42
TEST_SIZE = 0.2

## 1. Lecture des données

In [2]:
con = duckdb.connect(str(RAW_DB), read_only=True)
df = con.sql("SELECT * FROM housing").df()
con.close()

print(f"Shape : {df.shape}")
df.head()

Shape : (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 2. Séparation features / cible

In [3]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"X : {X.shape} — colonnes : {list(X.columns)}")
print(f"y : {y.shape} — nom : {y.name}")

X : (20640, 8) — colonnes : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
y : (20640,) — nom : MedHouseVal


## 3. Train/test split

- **80 % train / 20 % test**
- `random_state=42` pour la reproductibilité
- Pas de stratification : cible continue (régression)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f"X_train : {X_train.shape}    X_test : {X_test.shape}")
print(f"y_train : {y_train.shape}    y_test : {y_test.shape}")

X_train : (16512, 8)    X_test : (4128, 8)
y_train : (16512,)    y_test : (4128,)


In [5]:
y_train.describe()

count    16512.000000
mean         2.071947
std          1.156226
min          0.149990
25%          1.198000
50%          1.798500
75%          2.651250
max          5.000010
Name: MedHouseVal, dtype: float64

## 4. Standardisation des features

On **fit** le `StandardScaler` **sur `X_train` uniquement** (jamais sur le test → évite le *data leakage*), puis on transforme les deux ensembles avec le même scaler.

Le scaler est aussi sauvegardé sur disque pour être réutilisé tel quel en inférence (sinon, train/serve skew).

In [6]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

X_train_scaled.describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
MedInc,-6.519333e-17,1.00003,-1.775438,5.839268
HouseAge,-9.251859e-18,1.00003,-2.190766,1.856173
AveRooms,-1.981081e-16,1.00003,-1.904386,57.166552
AveBedrms,-1.707291e-16,1.00003,-1.762117,56.647272
Population,-2.151595e-19,1.00003,-1.251913,30.127428
AveOccup,4.936566e-17,1.00003,-0.207685,107.116447
Latitude,6.400995e-17,1.00003,-1.447697,2.951816
Longitude,1.753335e-15,1.00003,-2.377207,2.628794


## 5. Persistance

Deux tables dans `california_housing_processed.db` :
- `train` : `X_train_scaled` concaténé avec `y_train` (16 512 × 9)
- `test`  : `X_test_scaled`  concaténé avec `y_test`  (4 128 × 9)

Le scaler entraîné est sauvegardé dans `models/scaler.joblib`.

In [9]:
PROCESSED_DB.parent.mkdir(parents=True, exist_ok=True)
SCALER_PATH.parent.mkdir(parents=True, exist_ok=True)

train_df = X_train_scaled.assign(**{TARGET: y_train.values})
test_df = X_test_scaled.assign(**{TARGET: y_test.values})

con = duckdb.connect(str(PROCESSED_DB))
con.execute("CREATE OR REPLACE TABLE train AS SELECT * FROM train_df")
con.execute("CREATE OR REPLACE TABLE test  AS SELECT * FROM test_df")
con.close()

joblib.dump(scaler, SCALER_PATH)

print(f"Base   : {PROCESSED_DB.resolve()}")
print(f"Scaler : {SCALER_PATH.resolve()}")

Base   : /home/m/o/molivier/Mlops/projet/data/processed/california_housing_processed.db
Scaler : /home/m/o/molivier/Mlops/projet/models/scaler.joblib


In [10]:
# Vérification :
con = duckdb.connect(str(PROCESSED_DB), read_only=True)
print("Tables :", [r[0] for r in con.sql("SHOW TABLES").fetchall()])
print(f"train : {con.sql('SELECT COUNT(*) FROM train').fetchone()[0]} lignes")
print(f"test  : {con.sql('SELECT COUNT(*) FROM test').fetchone()[0]} lignes")
con.close()

Tables : ['test', 'train']
train : 16512 lignes
test  : 4128 lignes


### Comment réutiliser ces données dans la modélisation

```python
import duckdb
import joblib
from pathlib import Path

PROCESSED_DB = Path("../data/processed/california_housing_processed.db")
SCALER_PATH = Path("../models/scaler.joblib")
TARGET = "MedHouseVal"

con = duckdb.connect(str(PROCESSED_DB), read_only=True)
train = con.sql("SELECT * FROM train").df()
test  = con.sql("SELECT * FROM test").df()
con.close()

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_test,  y_test  = test.drop(columns=[TARGET]),  test[TARGET]

scaler = joblib.load(SCALER_PATH)  # à réutiliser pour scaler de nouvelles données en inférence
```